In [ ]:
# Google Colab Notebook: Model Training for Masked IP Detection

"""
MASKED IP DETECTION - MODEL TRAINING
=====================================
Train ML models to detect masked IPs
"""

# ============================================================================
# CELL 1: Setup
# ============================================================================

!pip install -q scikit-learn xgboost imbalanced-learn joblib

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

project_dir = '/content/drive/MyDrive/masked_ip_detection'

# ============================================================================
# CELL 2: Load and Prepare Data
# ============================================================================

print("Loading processed data...")

# Load the feature-engineered dataset
df = pd.read_csv(f'{project_dir}/data/processed/features_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")

# ============================================================================
# CELL 3: Feature Selection and Preparation
# ============================================================================

# Select features for training
feature_columns = [
    # IP basic features
    'ip_version', 'is_private', 'is_reserved', 'octet_1', 'octet_2',
    'octet_3', 'octet_4',
    
    # Geographic features
    'latitude', 'longitude', 'accuracy_radius',
    
    # ASN features
    'asn',
    
    # DNS features
    'has_ptr_record', 'ptr_contains_host',
    
    # Reputation features
    'in_tor_list', 'in_proxy_list', 'in_vpn_list',
    
    # Behavioral features (if available)
    'request_count', 'unique_user_agents',
]

# Filter to available columns
available_features = [col for col in feature_columns if col in df.columns]

print(f"Using {len(available_features)} features:")
print(available_features)

X = df[available_features]
y = df['label']

# Handle missing values
X = X.fillna(0)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

# ============================================================================
# CELL 4: Train-Test Split
# ============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# ============================================================================
# CELL 5: Handle Class Imbalance with SMOTE
# ============================================================================

print("\nApplying SMOTE to balance classes...")

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Original training set: {X_train.shape}")
print(f"Balanced training set: {X_train_balanced.shape}")
print(f"\nBalanced label distribution:\n{pd.Series(y_train_balanced).value_counts()}")

# ============================================================================
# CELL 6: Train Random Forest Model
# ============================================================================

print("\n" + "="*60)
print("TRAINING RANDOM FOREST MODEL")
print("="*60)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

rf_model.fit(X_train_balanced, y_train_balanced)

# Predictions
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluation
print("\nRandom Forest Performance:")
print(classification_report(y_test, y_pred_rf, target_names=['Legitimate', 'Masked']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")

# ============================================================================
# CELL 7: Train XGBoost Model
# ============================================================================

print("\n" + "="*60)
print("TRAINING XGBOOST MODEL")
print("="*60)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train_balanced, y_train_balanced)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluation
print("\nXGBoost Performance:")
print(classification_report(y_test, y_pred_xgb, target_names=['Legitimate', 'Masked']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")

# ============================================================================
# CELL 8: Train Gradient Boosting Model
# ============================================================================

print("\n" + "="*60)
print("TRAINING GRADIENT BOOSTING MODEL")
print("="*60)

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)

gb_model.fit(X_train_balanced, y_train_balanced)

# Predictions
y_pred_gb = gb_model.predict(X_test)
y_pred_proba_gb = gb_model.predict_proba(X_test)[:, 1]

# Evaluation
print("\nGradient Boosting Performance:")
print(classification_report(y_test, y_pred_gb, target_names=['Legitimate', 'Masked']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_gb):.4f}")

# ============================================================================
# CELL 9: Ensemble Model (Voting)
# ============================================================================

print("\n" + "="*60)
print("CREATING ENSEMBLE MODEL")
print("="*60)

# Weighted average of predictions
y_pred_proba_ensemble = (
    0.4 * y_pred_proba_rf + 
    0.4 * y_pred_proba_xgb + 
    0.2 * y_pred_proba_gb
)

y_pred_ensemble = (y_pred_proba_ensemble > 0.5).astype(int)

print("\nEnsemble Model Performance:")
print(classification_report(y_test, y_pred_ensemble, target_names=['Legitimate', 'Masked']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_ensemble):.4f}")

# ============================================================================
# CELL 10: Visualize Results
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Confusion Matrix - Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0])
axes[0, 0].set_title('Random Forest - Confusion Matrix')
axes[0, 0].set_xlabel('Predicted')
axes[0, 0].set_ylabel('Actual')

# 2. ROC Curves
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_pred_proba_xgb)
fpr_gb, tpr_gb, _ = roc_curve(y_test, y_pred_proba_gb)
fpr_ens, tpr_ens, _ = roc_curve(y_test, y_pred_proba_ensemble)

axes[0, 1].plot(fpr_rf, tpr_rf, label=f'RF (AUC={roc_auc_score(y_test, y_pred_proba_rf):.3f})')
axes[0, 1].plot(fpr_xgb, tpr_xgb, label=f'XGB (AUC={roc_auc_score(y_test, y_pred_proba_xgb):.3f})')
axes[0, 1].plot(fpr_gb, tpr_gb, label=f'GB (AUC={roc_auc_score(y_test, y_pred_proba_gb):.3f})')
axes[0, 1].plot(fpr_ens, tpr_ens, label=f'Ensemble (AUC={roc_auc_score(y_test, y_pred_proba_ensemble):.3f})', linewidth=2)
axes[0, 1].plot([0, 1], [0, 1], 'k--')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curves - All Models')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Feature Importance - Random Forest
feature_importance = pd.DataFrame({
    'feature': available_features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

axes[1, 0].barh(feature_importance['feature'], feature_importance['importance'])
axes[1, 0].set_xlabel('Importance')
axes[1, 0].set_title('Top 15 Feature Importances (Random Forest)')
axes[1, 0].invert_yaxis()

# 4. Model Comparison
model_scores = {
    'Random Forest': roc_auc_score(y_test, y_pred_proba_rf),
    'XGBoost': roc_auc_score(y_test, y_pred_proba_xgb),
    'Gradient Boosting': roc_auc_score(y_test, y_pred_proba_gb),
    'Ensemble': roc_auc_score(y_test, y_pred_proba_ensemble)
}

axes[1, 1].bar(model_scores.keys(), model_scores.values(), color=['blue', 'green', 'orange', 'red'])
axes[1, 1].set_ylabel('ROC-AUC Score')
axes[1, 1].set_title('Model Performance Comparison')
axes[1, 1].set_ylim([0.5, 1.0])
axes[1, 1].grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f'{project_dir}/model_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================================
# CELL 11: Save Models
# ============================================================================

print("\nSaving trained models...")

models_dir = f'{project_dir}/models'
os.makedirs(models_dir, exist_ok=True)

# Save models
joblib.dump(rf_model, f'{models_dir}/random_forest_model.pkl')
joblib.dump(xgb_model, f'{models_dir}/xgboost_model.pkl')
joblib.dump(gb_model, f'{models_dir}/gradient_boosting_model.pkl')

# Save feature names
joblib.dump(available_features, f'{models_dir}/feature_names.pkl')

# Save model metadata
metadata = {
    'trained_at': datetime.now().isoformat(),
    'feature_count': len(available_features),
    'training_samples': len(X_train_balanced),
    'test_samples': len(X_test),
    'models': {
        'random_forest': {
            'roc_auc': float(roc_auc_score(y_test, y_pred_proba_rf)),
            'accuracy': float((y_pred_rf == y_test).mean())
        },
        'xgboost': {
            'roc_auc': float(roc_auc_score(y_test, y_pred_proba_xgb)),
            'accuracy': float((y_pred_xgb == y_test).mean())
        },
        'gradient_boosting': {
            'roc_auc': float(roc_auc_score(y_test, y_pred_proba_gb)),
            'accuracy': float((y_pred_gb == y_test).mean())
        },
        'ensemble': {
            'roc_auc': float(roc_auc_score(y_test, y_pred_proba_ensemble)),
            'accuracy': float((y_pred_ensemble == y_test).mean())
        }
    }
}

import json
with open(f'{models_dir}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✓ Models saved successfully!")
print(f"✓ Location: {models_dir}")
print("\n" + "="*60)
print("MODEL TRAINING SUMMARY")
print("="*60)
for model_name, scores in metadata['models'].items():
    print(f"\n{model_name.upper()}:")
    print(f"  ROC-AUC: {scores['roc_auc']:.4f}")
    print(f"  Accuracy: {scores['accuracy']:.4f}")
print("="*60)